# CARLA Python API - Lab 4: Camera and LiDAR

This notebook has exactly 3 sections:
1. **RGB Camera** (theory + runnable demos + parameter guide)
2. **LiDAR** (theory + runnable demos + parameter guide)
3. **Exercises**

## CARLA docs
- Main docs: https://carla.readthedocs.io/en/latest/
- Sensors reference: https://carla.readthedocs.io/en/latest/ref_sensors/
- Python API: https://carla.readthedocs.io/en/latest/python_api/


In [2]:
import carla, time, random, cv2, queue, threading
import numpy as np


In [3]:
client = carla.Client("localhost", 2000)
client.set_timeout(10.0)
world = client.get_world()
spectator = world.get_spectator()


## 1. RGB Camera: how it works + configurable parameters

`spawn_camera(world, attach_to, transform, width=640, height=360, fov=95, tick=0.05)` mounts an RGB sensor on a parent actor.

What each argument controls:
- `world`: CARLA world used to fetch blueprint and spawn the camera actor.
- `attach_to`: parent actor (usually the ego vehicle).
- `transform`: camera mount pose relative to parent (`x, y, z, pitch, yaw, roll`).
- `width` -> `image_size_x`: image width in pixels.
- `height` -> `image_size_y`: image height in pixels.
- `fov`: horizontal field of view in degrees (smaller = zoom-like, larger = wide-angle).
- `tick` -> `sensor_tick`: seconds between frames (`0.05` ~= 20 FPS max).

Data pipeline in callback:
1. CARLA provides `carla.Image`.
2. `raw_data` is BGRA bytes.
3. Convert to OpenCV BGR with `image_to_bgr`.

Camera parameters you should tune first in practice:
- Resolution (`width`, `height`): visual detail vs compute cost.
- `fov`: scene coverage vs geometric distortion.
- `sensor_tick`: temporal smoothness vs processing load.
- Mount `transform`: what the model can actually see.

Blueprint-level camera attributes can include many additional options (for example post-processing, exposure, bloom, lens effects) depending on CARLA version.
Run the next cell to inspect the exact attributes available in your installation.


### Camera parameter reference from your installed CARLA build

Run the next cell to print all camera blueprint attributes, current value, mutability, and recommended values.


In [4]:
def move_spectator_to(transform, spectator, distance=7.0, z=3.0, pitch=-15.0):
    back = transform.location - transform.get_forward_vector() * distance
    loc = carla.Location(back.x, back.y, back.z + z)
    rot = carla.Rotation(pitch=pitch, yaw=transform.rotation.yaw, roll=0.0)
    spectator.set_transform(carla.Transform(loc, rot))


def spawn_vehicle(world, spawn_index=0, vehicle_filter="vehicle.tesla.model3", autopilot=False):
    points = world.get_map().get_spawn_points()
    if not points:
        raise RuntimeError("No spawn points found")
    bps = world.get_blueprint_library().filter(vehicle_filter)
    if not bps:
        bps = world.get_blueprint_library().filter("vehicle.*")
    for k in range(len(points)):
        actor = world.try_spawn_actor(random.choice(bps), points[(spawn_index + k) % len(points)])
        if actor is not None:
            actor.set_autopilot(autopilot)
            return actor
    raise RuntimeError("Could not spawn vehicle")


def spawn_vehicle_ahead(world, ref_vehicle, distances=(20.0, 28.0, 36.0), same_lane_first=True, retries=6):
    ego_tf = ref_vehicle.get_transform()
    ego_loc = ego_tf.location
    fwd = ego_tf.get_forward_vector()
    right = ego_tf.get_right_vector()

    bps = world.get_blueprint_library().filter("vehicle.*")
    if not bps:
        raise RuntimeError("No vehicle blueprints found")

    # Phase 1: try exact transforms straight ahead of ego so the target is visually in front.
    direct = sorted(set(float(d) for d in distances))
    direct += [d + 4.0 for d in direct]
    for d in direct:
        loc = ego_loc + fwd * d + right * 0.0
        loc.z += 0.30
        tf = carla.Transform(loc, ego_tf.rotation)
        actor = world.try_spawn_actor(random.choice(bps), tf)
        if actor is not None:
            actor.set_autopilot(False)
            return actor

    # Phase 2: fallback to road waypoints in front.
    road_map = world.get_map()
    ego_wp = road_map.get_waypoint(ego_loc, project_to_road=True, lane_type=carla.LaneType.Driving)

    wp_candidates = []
    d_pool = sorted(set(float(d) for d in distances) | set(float(d) + 8.0 for d in distances))
    for d in d_pool:
        try:
            cands = ego_wp.next(float(d))
        except RuntimeError:
            cands = []
        if same_lane_first:
            cands = sorted(
                cands,
                key=lambda w: (
                    w.road_id != ego_wp.road_id,
                    w.lane_id != ego_wp.lane_id,
                    abs(w.lane_id - ego_wp.lane_id),
                ),
            )
        wp_candidates.extend(cands)

    for _ in range(max(1, int(retries))):
        for wp in wp_candidates:
            actor = world.try_spawn_actor(random.choice(bps), wp.transform)
            if actor is not None:
                actor.set_autopilot(False)
                return actor
        world.tick()
        time.sleep(0.02)

    # Phase 3: fallback to an already existing vehicle in front.
    best = None
    best_d = float("inf")
    for a in world.get_actors().filter("vehicle.*"):
        if a.id == ref_vehicle.id:
            continue
        loc = a.get_transform().location
        rel = loc - ego_loc
        d_fwd = rel.x * fwd.x + rel.y * fwd.y
        d_lat = abs(rel.x * (-fwd.y) + rel.y * fwd.x)
        if d_fwd > 6.0 and d_lat < 4.0 and d_fwd < best_d:
            best = a
            best_d = d_fwd

    if best is not None:
        try:
            best.set_autopilot(False)
        except RuntimeError:
            pass
        return best

    raise RuntimeError("Could not spawn or find a target vehicle ahead")


def spawn_camera(world, attach_to, transform, width=640, height=360, fov=95, tick=0.05):
    bp = world.get_blueprint_library().find("sensor.camera.rgb")
    bp.set_attribute("image_size_x", str(width))
    bp.set_attribute("image_size_y", str(height))
    bp.set_attribute("fov", str(fov))
    bp.set_attribute("sensor_tick", str(tick))
    return world.spawn_actor(bp, transform, attach_to=attach_to)


def spawn_lidar(world, attach_to, transform, channels=32, points_per_second=56000, rotation_frequency=20, range_m=35):
    bp = world.get_blueprint_library().find("sensor.lidar.ray_cast")
    bp.set_attribute("channels", str(channels))
    bp.set_attribute("points_per_second", str(points_per_second))
    bp.set_attribute("rotation_frequency", str(rotation_frequency))
    bp.set_attribute("range", str(range_m))
    return world.spawn_actor(bp, transform, attach_to=attach_to)


def image_to_bgr(image):
    arr = np.frombuffer(image.raw_data, dtype=np.uint8)
    arr = np.reshape(arr, (image.height, image.width, 4))
    return arr[:, :, :3].copy()


def lidar_to_numpy(measurement):
    pts = np.frombuffer(measurement.raw_data, dtype=np.float32)
    return np.reshape(pts, (-1, 4))


def safe_destroy(actors):
    for a in actors:
        if a is not None:
            try:
                a.destroy()
            except RuntimeError:
                pass


In [5]:
# Camera blueprint attributes (compatible across CARLA versions)
bp = world.get_blueprint_library().find("sensor.camera.rgb")
print("camera blueprint id:", bp.id)


def _attr_value(attr):
    # Some CARLA builds expose default_value, others do not.
    for name in ("default_value", "value", "default"):
        if hasattr(attr, name):
            try:
                v = getattr(attr, name)
                if v is not None:
                    return str(v)
            except Exception:
                pass
    try:
        return str(attr)
    except Exception:
        return "<n/a>"


for attr in bp:
    attr_id = getattr(attr, "id", "<unknown>")
    attr_type = str(getattr(attr, "type", ""))
    modifiable = bool(getattr(attr, "is_modifiable", False))
    rec = [str(v) for v in list(getattr(attr, "recommended_values", []))]
    rec_show = ", ".join(rec[:8]) + (" ..." if len(rec) > 8 else "")
    value = _attr_value(attr)
    print(f"{attr_id:26} type={attr_type:12} value={value:>10} modifiable={modifiable} rec=[{rec_show}]")


camera blueprint id: sensor.camera.rgb
role_name                  type=String       value=ActorAttribute(id=role_name,type=str,value=front) modifiable=True rec=[front, back, left, right, front_left, front_right, back_left, back_right]
ros_name                   type=String       value=ActorAttribute(id=ros_name,type=str,value=sensor.camera.rgb) modifiable=True rec=[sensor.camera.rgb]
chromatic_aberration_offset type=Float        value=ActorAttribute(id=chromatic_aberration_offset,type=float,value=0) modifiable=True rec=[0.0]
sensor_tick                type=Float        value=ActorAttribute(id=sensor_tick,type=float,value=0) modifiable=True rec=[0.0]
fstop                      type=Float        value=ActorAttribute(id=fstop,type=float,value=1.4) modifiable=True rec=[1.4]
image_size_x               type=Int          value=ActorAttribute(id=image_size_x,type=int,value=800) modifiable=True rec=[800]
image_size_y               type=Int          value=ActorAttribute(id=image_size_y,type=int,

### Camera runnable demo

This demo spawns a driving vehicle and an attached camera so you can see mounting + streaming in action.


In [ ]:
# Demo - RGB camera mounted on a moving vehicle
# Press 'q' in the image window to stop early.
vehicle = None
camera = None
latest = {"frame": None}
DURATION_SECONDS = 30.0

try:
    vehicle = spawn_vehicle(world, spawn_index=12, autopilot=True)

    # Mounted at windshield position, slightly pitched down.
    cam_tf = carla.Transform(
        carla.Location(x=1.8, z=1.4),
        carla.Rotation(pitch=-6.0),
    )
    camera = spawn_camera(world, vehicle, cam_tf, width=800, height=450, fov=95, tick=0.05)

    def on_image(image):
        latest["frame"] = image_to_bgr(image)

    camera.listen(on_image)

    t0 = time.time()
    while time.time() - t0 < DURATION_SECONDS:
        frame = latest["frame"]
        if frame is not None:
            cv2.imshow("Section 1 Demo - RGB Camera", frame)
        move_spectator_to(vehicle.get_transform(), spectator, distance=8.0, z=3.0)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
        world.tick()
        time.sleep(0.02)
finally:
    if camera is not None:
        camera.stop()
    safe_destroy([camera, vehicle])
    cv2.destroyAllWindows()


: 

### Camera parameter effect demo (FOV comparison)

Run this cell to compare two cameras mounted on the same vehicle:
- left panel: `fov=60` (narrow)
- right panel: `fov=110` (wide)

This makes the effect of `fov` immediately visible.


In [ ]:
# Demo - Compare narrow vs wide FOV on the same driving vehicle
# Press 'q' to stop.
vehicle = None
cameras = []
frame_queues = {}
latest = {}
DURATION_SECONDS = 20.0

configs = {
    "fov60": {"fov": 60, "width": 640, "height": 360, "tick": 0.05},
    "fov110": {"fov": 110, "width": 640, "height": 360, "tick": 0.05},
}

try:
    vehicle = spawn_vehicle(world, spawn_index=14, autopilot=True)
    cam_tf = carla.Transform(carla.Location(x=1.8, z=1.4), carla.Rotation(pitch=-6.0))

    def make_cb(name):
        q = frame_queues[name]

        def _cb(image):
            frame = image_to_bgr(image)
            if q.full():
                try:
                    q.get_nowait()
                except queue.Empty:
                    pass
            try:
                q.put_nowait(frame)
            except queue.Full:
                pass

        return _cb

    for name, cfg in configs.items():
        cam = spawn_camera(
            world,
            vehicle,
            cam_tf,
            width=cfg["width"],
            height=cfg["height"],
            fov=cfg["fov"],
            tick=cfg["tick"],
        )
        cameras.append(cam)
        frame_queues[name] = queue.Queue(maxsize=1)
        latest[name] = np.zeros((cfg["height"], cfg["width"], 3), dtype=np.uint8)
        cam.listen(make_cb(name))

    t0 = time.time()
    while time.time() - t0 < DURATION_SECONDS:
        for name, q in frame_queues.items():
            try:
                latest[name] = q.get_nowait()
            except queue.Empty:
                pass

        left = latest["fov60"].copy()
        right = latest["fov110"].copy()
        cv2.putText(left, "FOV 60", (14, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 255), 2)
        cv2.putText(right, "FOV 110", (14, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 255), 2)
        panel = np.hstack([left, right])

        cv2.imshow("Section 1 Demo - Camera FOV Comparison", panel)
        move_spectator_to(vehicle.get_transform(), spectator, distance=8.0, z=3.0)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
        world.tick()
        time.sleep(0.02)
finally:
    for cam in cameras:
        cam.stop()
    safe_destroy(cameras + [vehicle])
    cv2.destroyAllWindows()


## 2. LiDAR: how it works + configurable parameters

`spawn_lidar(world, attach_to, transform, channels=32, points_per_second=56000, rotation_frequency=20, range_m=35)` mounts a ray-cast LiDAR.

What each argument controls:
- `world`: CARLA world used to fetch blueprint and spawn the LiDAR actor.
- `attach_to`: parent actor (usually the ego vehicle).
- `transform`: LiDAR mount pose relative to parent.
- `channels`: number of vertical laser layers.
- `points_per_second`: total points generated every second.
- `rotation_frequency`: horizontal rotations per second (Hz).
- `range_m` -> `range`: maximum sensing distance in meters.

LiDAR callback data format:
- `measurement.raw_data` is a flat `float32` array.
- reshape to `(-1, 4)` -> columns are `[x, y, z, intensity]` in sensor-local frame.

Core tuning trade-offs:
- More `channels` and `points_per_second` => denser cloud, higher compute cost.
- Higher `rotation_frequency` => faster updates, fewer points per revolution at fixed budget.
- Larger `range` => farther sensing, potentially noisier/fewer near points depending on setup.

Blueprint-level LiDAR attributes can include additional controls (for example upper/lower FOV, dropoff/noise, atmosphere attenuation), depending on CARLA version.
Run the reference cell to inspect exactly what your build exposes.


### LiDAR parameter reference from your installed CARLA build

Run the next cell to print all LiDAR blueprint attributes, current value, mutability, and recommended values.


In [16]:
# LiDAR blueprint attributes (compatible across CARLA versions)
bp = world.get_blueprint_library().find("sensor.lidar.ray_cast")
print("lidar blueprint id:", bp.id)


def _attr_value(attr):
    for name in ("default_value", "value", "default"):
        if hasattr(attr, name):
            try:
                v = getattr(attr, name)
                if v is not None:
                    return str(v)
            except Exception:
                pass
    try:
        return str(attr)
    except Exception:
        return "<n/a>"


for attr in bp:
    attr_id = getattr(attr, "id", "<unknown>")
    attr_type = str(getattr(attr, "type", ""))
    modifiable = bool(getattr(attr, "is_modifiable", False))
    rec = [str(v) for v in list(getattr(attr, "recommended_values", []))]
    rec_show = ", ".join(rec[:8]) + (" ..." if len(rec) > 8 else "")
    value = _attr_value(attr)
    print(f"{attr_id:26} type={attr_type:12} value={value:>10} modifiable={modifiable} rec=[{rec_show}]")


lidar blueprint id: sensor.lidar.ray_cast
atmosphere_attenuation_rate type=Float        value=ActorAttribute(id=atmosphere_attenuation_rate,type=float,value=0.004) modifiable=True rec=[0.004]
role_name                  type=String       value=ActorAttribute(id=role_name,type=str,value=front) modifiable=True rec=[front, back, left, right, front_left, front_right, back_left, back_right]
noise_seed                 type=Int          value=ActorAttribute(id=noise_seed,type=int,value=0) modifiable=True rec=[0]
upper_fov                  type=Float        value=ActorAttribute(id=upper_fov,type=float,value=10) modifiable=True rec=[10.0]
ros_name                   type=String       value=ActorAttribute(id=ros_name,type=str,value=sensor.lidar.ray_cast) modifiable=True rec=[sensor.lidar.ray_cast]
sensor_tick                type=Float        value=ActorAttribute(id=sensor_tick,type=float,value=0) modifiable=True rec=[0.0]
channels                   type=Int          value=ActorAttribute(id=channel

### LiDAR runnable demo

This demo now shows two things at the same time:
- a visible **TARGET** marker above the front vehicle in the world view;
- a live **top-view point cloud window** (all points in gray, front-corridor points in yellow, nearest point in red).

It also applies smooth throttle/brake control based on LiDAR front distance.


In [ ]:
# Demo - LiDAR point cloud + smooth braking control
# Press 'q' in the point-cloud window to stop early.
vehicle = None
target = None
lidar = None
state = {"d_front_raw": None, "d_front": None, "pcd_img": None, "pcd_prev": None}
DURATION_SECONDS = 26.0

try:
    vehicle = spawn_vehicle(world, spawn_index=25, autopilot=False)

    try:
        target = spawn_vehicle_ahead(world, vehicle, distances=(16.0, 20.0, 24.0), same_lane_first=True, retries=10)

        # Reposition target exactly in front of ego for a clearer didactic setup.
        ego_tf = vehicle.get_transform()
        desired_loc = ego_tf.location + ego_tf.get_forward_vector() * 20.0
        desired_loc.z += 0.30
        target.set_transform(carla.Transform(desired_loc, ego_tf.rotation))

        target.apply_control(carla.VehicleControl(throttle=0.0, brake=1.0, hand_brake=True))
    except RuntimeError as e:
        target = None
        print("Warning:", e)
        print("Continuing demo without a dedicated front target.")

    lidar_tf = carla.Transform(carla.Location(x=1.8, z=2.2))
    lidar = spawn_lidar(
        world,
        vehicle,
        lidar_tf,
        channels=64,
        points_per_second=120000,
        rotation_frequency=30,
        range_m=50,
    )

    canvas_h, canvas_w = 760, 760
    scale = 16.0
    origin_x, origin_y = canvas_w // 2, canvas_h - 42

    def make_canvas():
        img = np.zeros((canvas_h, canvas_w, 3), dtype=np.uint8)
        cv2.line(img, (origin_x, origin_y), (origin_x, 20), (45, 45, 45), 1)
        cv2.line(img, (35, origin_y), (canvas_w - 35, origin_y), (45, 45, 45), 1)
        cv2.putText(img, "ego", (origin_x + 6, origin_y - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (180, 180, 180), 1)
        cv2.putText(img, "x forward", (origin_x + 10, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (180, 180, 180), 1)
        return img

    def on_lidar(measurement):
        pts = lidar_to_numpy(measurement)
        x = pts[:, 0]
        y = pts[:, 1]
        z = pts[:, 2]

        # Larger visible area for didactic point-cloud visualization.
        vis = (x > -6.0) & (x < 40.0) & (np.abs(y) < 18.0) & (z > -2.5) & (z < 2.8)
        x_vis = x[vis]
        y_vis = y[vis]

        # Front corridor used for distance estimation.
        front = (x > 0.8) & (x < 32.0) & (np.abs(y) < 2.0) & (z > -1.7) & (z < 2.2)
        if np.any(front):
            d = np.sqrt(x[front] ** 2 + y[front] ** 2)
            # Percentile is more stable than min and reduces sparkly outliers.
            raw = float(np.percentile(d, 10))
            state["d_front_raw"] = raw
            prev = state["d_front"]
            state["d_front"] = raw if prev is None else (0.82 * prev + 0.18 * raw)
        else:
            state["d_front_raw"] = None
            state["d_front"] = None

        img = make_canvas()

        if x_vis.size > 0:
            px = (origin_x + y_vis * scale).astype(np.int32)
            py = (origin_y - x_vis * scale).astype(np.int32)
            ok = (px >= 0) & (px < canvas_w) & (py >= 0) & (py < canvas_h)
            img[py[ok], px[ok]] = (175, 175, 175)

        xf = x[front]
        yf = y[front]
        if xf.size > 0:
            pxf = (origin_x + yf * scale).astype(np.int32)
            pyf = (origin_y - xf * scale).astype(np.int32)
            okf = (pxf >= 0) & (pxf < canvas_w) & (pyf >= 0) & (pyf < canvas_h)
            img[pyf[okf], pxf[okf]] = (0, 220, 255)

            d = np.sqrt(xf ** 2 + yf ** 2)
            idx = int(np.argmin(d))
            cx = int(origin_x + yf[idx] * scale)
            cy = int(origin_y - xf[idx] * scale)
            if 0 <= cx < canvas_w and 0 <= cy < canvas_h:
                cv2.circle(img, (cx, cy), 5, (0, 0, 255), -1)

        # Draw front corridor in top-view.
        x0, x1 = 0.8, 32.0
        y0, y1 = -2.0, 2.0
        p1 = (int(origin_x + y0 * scale), int(origin_y - x0 * scale))
        p2 = (int(origin_x + y1 * scale), int(origin_y - x0 * scale))
        p3 = (int(origin_x + y1 * scale), int(origin_y - x1 * scale))
        p4 = (int(origin_x + y0 * scale), int(origin_y - x1 * scale))
        cv2.line(img, p1, p2, (80, 140, 255), 1)
        cv2.line(img, p2, p3, (80, 140, 255), 1)
        cv2.line(img, p3, p4, (80, 140, 255), 1)
        cv2.line(img, p4, p1, (80, 140, 255), 1)

        if state["d_front"] is not None:
            cv2.putText(
                img,
                "d_est={:.2f} m".format(state["d_front"]),
                (20, 30),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0, 255, 255),
                2,
            )

        prev_img = state["pcd_prev"]
        if prev_img is not None:
            img = cv2.addWeighted(prev_img, 0.35, img, 0.65, 0.0)

        state["pcd_img"] = img
        state["pcd_prev"] = img

    lidar.listen(on_lidar)

    stop_d = 7.5
    slow_d = 16.0
    max_throttle = 0.35
    throttle = 0.0
    brake = 0.0

    last_print = 0.0
    t0 = time.time()

    while time.time() - t0 < DURATION_SECONDS:
        d_est = state["d_front"]

        if d_est is None:
            tgt_throttle = max_throttle
            tgt_brake = 0.0
        elif d_est <= stop_d:
            tgt_throttle = 0.0
            tgt_brake = 0.90
        elif d_est < slow_d:
            alpha = (d_est - stop_d) / (slow_d - stop_d)
            tgt_throttle = 0.08 + (max_throttle - 0.08) * alpha
            tgt_brake = 0.45 * (1.0 - alpha)
        else:
            tgt_throttle = max_throttle
            tgt_brake = 0.0

        throttle = 0.84 * throttle + 0.16 * tgt_throttle
        brake = 0.84 * brake + 0.16 * tgt_brake

        if throttle < 0.02:
            throttle = 0.0
        if brake < 0.02:
            brake = 0.0

        vehicle.apply_control(
            carla.VehicleControl(
                throttle=float(np.clip(throttle, 0.0, 1.0)),
                brake=float(np.clip(brake, 0.0, 1.0)),
                steer=0.0,
            )
        )

        d_gt = None
        if target is not None:
            target_loc = target.get_transform().location
            ego_loc = vehicle.get_transform().location
            d_gt = ego_loc.distance(target_loc)
            world.debug.draw_string(
                target_loc + carla.Location(z=1.8),
                "TARGET",
                life_time=0.08,
                color=carla.Color(0, 255, 255),
            )

        pcd_img = state["pcd_img"]
        if pcd_img is not None:
            frame = pcd_img.copy()
            if d_gt is not None:
                cv2.putText(
                    frame,
                    "d_gt={:.2f} m".format(d_gt),
                    (20, 58),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    (80, 220, 80),
                    2,
                )
            cv2.imshow("Section 2 Demo - LiDAR Point Cloud (top view)", frame)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

        if time.time() - last_print > 1.0:
            print(
                "d_raw={} d_est={} d_gt={} throttle={:.2f} brake={:.2f}".format(
                    "{:.2f}".format(state["d_front_raw"]) if state["d_front_raw"] is not None else "None",
                    "{:.2f}".format(d_est) if d_est is not None else "None",
                    "{:.2f}".format(d_gt) if d_gt is not None else "None",
                    throttle,
                    brake,
                )
            )
            last_print = time.time()

        move_spectator_to(vehicle.get_transform(), spectator, distance=11.0, z=5.0, pitch=-18.0)
        world.tick()
        time.sleep(0.015)
finally:
    if lidar is not None:
        lidar.stop()
    safe_destroy([lidar, target, vehicle])
    cv2.destroyAllWindows()


d_raw=16.65 d_est=None d_gt=20.00 throttle=0.06 brake=0.00
d_raw=15.93 d_est=16.18 d_gt=20.00 throttle=0.34 brake=0.00
d_raw=14.47 d_est=14.84 d_gt=18.61 throttle=0.33 brake=0.00
d_raw=11.08 d_est=11.03 d_gt=15.02 throttle=0.26 brake=0.14
d_raw=None d_est=None d_gt=11.28 throttle=0.32 brake=0.05
d_raw=None d_est=3.49 d_gt=7.54 throttle=0.04 brake=0.80
d_raw=2.67 d_est=2.72 d_gt=6.72 throttle=0.23 brake=0.30
d_raw=2.70 d_est=2.68 d_gt=6.72 throttle=0.20 brake=0.40
d_raw=2.69 d_est=2.77 d_gt=6.72 throttle=0.10 brake=0.64
d_raw=2.69 d_est=2.72 d_gt=6.72 throttle=0.04 brake=0.79
d_raw=2.66 d_est=None d_gt=6.72 throttle=0.12 brake=0.59
d_raw=2.70 d_est=2.70 d_gt=6.72 throttle=0.16 brake=0.48
d_raw=2.68 d_est=None d_gt=6.72 throttle=0.13 brake=0.55
d_raw=None d_est=None d_gt=6.72 throttle=0.23 brake=0.32
d_raw=2.69 d_est=None d_gt=6.72 throttle=0.17 brake=0.47
d_raw=2.69 d_est=2.68 d_gt=6.72 throttle=0.15 brake=0.51
d_raw=2.68 d_est=2.69 d_gt=6.72 throttle=0.07 brake=0.71
d_raw=None d_est=2.

## 3. Exercises (camera + LiDAR only)

Complete these in order: 1 -> 2 -> 3.


### Exercise 1: Streamer cockpit (4 cameras)

Build 4 synchronized views:
- front windshield
- left mirror
- right mirror
- rear view

Docs:
- RGB camera: https://carla.readthedocs.io/en/latest/ref_sensors/#rgb-camera
- Transform API: https://carla.readthedocs.io/en/latest/python_api/#carlatransform


In [6]:
# TEMPLATE - Exercise 1
vehicle = None
cameras = []
frame_queues = {}
latest_frames = {}

specs = {
    "front": carla.Transform(carla.Location(x=1.8, z=1.4), carla.Rotation(pitch=-6.0)),
    "left": carla.Transform(carla.Location(x=0.1, y=-0.75, z=1.3), carla.Rotation(yaw=-145.0)),
    "right": carla.Transform(carla.Location(x=0.1, y=0.75, z=1.3), carla.Rotation(yaw=145.0)),
    "rear": carla.Transform(carla.Location(x=-2.1, z=1.3), carla.Rotation(yaw=180.0)),
}

stop_event = threading.Event()
quit_event = threading.Event()
viewer_thread = None

try:
    # STEP 1: spawn ego vehicle with autopilot
    vehicle = spawn_vehicle(world, spawn_index=14, autopilot=True)
    cam_tf = carla.Transform(carla.Location(x=1.8, z=1.4), carla.Rotation(pitch=-6.0))

    # STEP 2: spawn one camera per view and one Queue(maxsize=1) per camera
    for name, cfg in specs.items():
        cam = spawn_camera(
            world,
            vehicle,
            cam_tf[cfg],
            width=800,
            height=450,
            fov=95,
            tick=0.05,
        )
        cameras.append(cam)
        frame_queues[name] = queue.Queue(maxsize=1)
        latest_frames[name] = np.zeros((450, 800, 3), dtype=np.uint8)
        cam.listen(make_cb(name))

    t0 = time.time()
    while time.time() - t0 < DURATION_SECONDS:
        for name, q in frame_queues.items():
            try:
                latest[name] = q.get_nowait()
            except queue.Empty:
                pass

        left = latest_frames["left"].copy()
        right = latest_frames["right"].copy()
        rear = latest_frames["rear"].copy()
        front = latest_frames["front"].copy()
        cv2.putText(left, "left", (14, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 255), 2)
        cv2.putText(right, "right", (14, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 255), 2)
        cv2.putText(rear, "rear", (14, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 255), 2)
        cv2.putText(front, "front", (14, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 255), 2)
        panel = np.hstack([left, right, rear, front])

        cv2.imshow("Section 1 Demo - Camera FOV Comparison", panel)
        move_spectator_to(vehicle.get_transform(), spectator, distance=8.0, z=3.0)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
        world.tick()
        time.sleep(0.02)

    # STEP 3: callback factory per camera name
    # - convert image to BGR
    # - push frame to queue without blocking (drop old frame when full)
    # TODO

    # STEP 4: start a dedicated viewer thread
    # - thread reads latest frames and calls cv2.imshow/cv2.waitKey
    # - main thread only runs simulation tick + spectator
    # TODO
    pass
finally:
    for cam in cameras:
        cam.stop()
    safe_destroy(cameras + [vehicle])
    cv2.destroyAllWindows()
    pass


: 

### Exercise 2: Camera HUD + edge vision dashboard

Build a front-camera dashboard with two panels:
- left: RGB feed with HUD overlays (speed + crosshair)
- right: edge view (Canny)

Docs:
- RGB camera: https://carla.readthedocs.io/en/latest/ref_sensors/#rgb-camera
- Vehicle velocity: https://carla.readthedocs.io/en/latest/python_api/


In [7]:
# TEMPLATE - Exercise 2
vehicle = None
camera = None
state = {"frame": None}

try:
    vehicle = spawn_vehicle(world, spawn_index=12, autopilot=True)

    # Mounted at windshield position, slightly pitched down.
    cam_tf = carla.Transform(
        carla.Location(x=1.8, z=1.4),
        carla.Rotation(pitch=-6.0),
    )
    camera = spawn_camera(world, vehicle, cam_tf, width=800, height=450, fov=95, tick=0.05)

    def on_image(image):
        state["frame"] = image_to_bgr(image)

    camera.listen(on_image)

    while True:
        frame = state["frame"]
        if frame is not None:
            # STEP 3: compute speed in km/h from vehicle velocity
            v = vehicle.get_velocity(); 
            speed = 3.6 * np.sqrt(v.x*v.x + v.y*v.y + v.z*v.z)

            # STEP 4: draw HUD overlays on RGB frame (crosshair + speed text)
            cv2.line(frame, (400, 225 - 20), (400, 225 + 20), (0, 255, 255), 2)
            cv2.line(frame, (400 - 20, 225), (400 + 20, 225), (0, 255, 255), 2)
            cv2.putText(frame, "{:.1f} km/h".format(speed), (14, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 255), 2)
            
            # STEP 5: compute Canny edges and stack [RGB_HUD | edges_bgr]
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            edges = cv2.Canny(gray, 80, 160)
            edges_bgr = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)
            panel = np.hstack([frame, edges_bgr])

            # STEP 6: show result with cv2.imshow and allow quit with 'q'
            cv2.imshow("Section 1 Demo - RGB Camera with HUD", panel)
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break
            
            pass

        move_spectator_to(vehicle.get_transform(), spectator, distance=8.0, z=3.0)
        world.tick()
        time.sleep(0.02)
except KeyboardInterrupt:
    pass
finally:
    if camera is not None:
        camera.stop()
    safe_destroy([camera, vehicle])
    cv2.destroyAllWindows()


### Exercise 3: LiDAR basic safety brake

Attach a LiDAR, estimate the nearest obstacle in front, and control the car with simple thresholds.

To avoid false detections, use:
- a narrower front corridor,
- a minimum number of points,
- a short hold time before setting distance back to `None`.

Docs:
- LiDAR sensor: https://carla.readthedocs.io/en/latest/ref_sensors/#lidar-sensor
- VehicleControl: https://carla.readthedocs.io/en/latest/python_api/#carlavehiclecontrol


In [ ]:
# TEMPLATE - Exercise 3
vehicle = None
target = None
lidar = None
state = {"d_front": None, "last_seen_ts": 0.0}

try:
    # STEP 1: spawn ego vehicle
    # TODO
    vehicle = spawn_vehicle(world, spawn_index=12, autopilot=True)

    # STEP 2: spawn a target in front and keep it stopped
    target = spawn_vehicle_ahead(world, vehicle, distances=(18.0, 22.0, 26.0), retries=8)
    target.apply_control(carla.VehicleControl(throttle=0.0, brake=1.0, hand_brake=True))
    # TODO
    

    # STEP 3: attach LiDAR to ego
    # TODO
    lidar = spawn_lidar(
        world,
        vehicle,
        carla.Transform(carla.Location(x=1.8, z=2.2)),
        channels=64,
        points_per_second=120000,
        rotation_frequency=30,
        range_m=50,
    )

    def on_lidar(measurement):
        pts = lidar_to_numpy(measurement)
        x = pts[:, 0]
        y = pts[:, 1]
        z = pts[:, 2]

        # Larger visible area for didactic point-cloud visualization.
        vis = (x > -6.0) & (x < 40.0) & (np.abs(y) < 18.0) & (z > -2.5) & (z < 2.8)
        x_vis = x[vis]
        y_vis = y[vis]

        # Front corridor used for distance estimation.
        front = (x > 0.8) & (x < 32.0) & (np.abs(y) < 2.0) & (z > -1.7) & (z < 2.2)
        if np.any(front):
            d = np.sqrt(x[front] ** 2 + y[front] ** 2)
            # Percentile is more stable than min and reduces sparkly outliers.
            raw = float(np.percentile(d, 10))
            state["d_front_raw"] = raw
            prev = state["d_front"]
            state["d_front"] = raw if prev is None else (0.82 * prev + 0.18 * raw)
        else:
            state["d_front_raw"] = None
            state["d_front"] = None


    lidar.listen(on_lidar)

    last_print = 0.0
    

    while True:
        d = state["d_front"]

        # STEP 7: simple threshold controller
        # - far or no obstacle: throttle=0.35, brake=0.0
        # - medium distance: throttle=0.12, brake=0.20
        # - close obstacle: throttle=0.0, brake=1.0
        # TODO
        if d is None:
            throttle = 0.35
            brake = 0.0
        elif d < 7.5:
            throttle = 0.0
            brake = 1.0
        elif d < 16.0:
            throttle = 0.12
            brake = 0.20
        else:
            throttle = 0.35
            brake = 0.0
        vehicle.apply_control(
            carla.VehicleControl(
                throttle=float(np.clip(throttle, 0.0, 1.0)),
                brake=float(np.clip(brake, 0.0, 1.0)),
                steer=0.0,
            )
        )   
        

        # STEP 8 (optional): draw current distance + TARGET label
        # TODO
        if target is not None:
            target_loc = target.get_transform().location
            world.debug.draw_string(
                target_loc + carla.Location(z=1.8),
                "TARGET",
                life_time=0.08,
                color=carla.Color(0, 255, 255),
            )



        # STEP 9: print distance every 1 second
        # TODO
        if time.time() - last_print > 1.0:
            print("d_front={}".format("{:.2f} m".format(d) if d is not None else "None"))
            last_print = time.time()
            

        move_spectator_to(vehicle.get_transform(), spectator, distance=10.0, z=4.0, pitch=-16.0)
        world.tick()
        time.sleep(0.03)
except KeyboardInterrupt:
    pass
finally:
    if lidar is not None:
        lidar.stop()
    safe_destroy([lidar, target, vehicle])


d_front=None
d_front=14.87 m
d_front=11.21 m
d_front=None
d_front=2.38 m
d_front=1.78 m
d_front=None
d_front=1.77 m
d_front=1.77 m
d_front=1.76 m
d_front=1.78 m
d_front=1.78 m
d_front=1.78 m
d_front=1.79 m
d_front=1.82 m
d_front=1.84 m
d_front=None
d_front=1.79 m
d_front=None
d_front=1.77 m
d_front=1.77 m
d_front=1.79 m
d_front=1.79 m
d_front=1.76 m
d_front=None
d_front=None
d_front=1.78 m
d_front=1.79 m
d_front=1.78 m
d_front=1.78 m
d_front=1.78 m
d_front=1.79 m
d_front=1.77 m
d_front=1.79 m
d_front=None
d_front=1.78 m
d_front=1.81 m
d_front=1.78 m
d_front=1.78 m
d_front=1.79 m
d_front=None
d_front=1.78 m
d_front=1.80 m
d_front=1.79 m
